# Basic tour of the Bayesian Optimization package

This is a constrained global optimization package built upon bayesian inference and gaussian process, that attempts to find the maximum value of an unknown function in as few iterations as possible. This technique is particularly suited for optimization of high cost functions, situations where the balance between exploration and exploitation is important.

Bayesian optimization works by constructing a posterior distribution of functions (gaussian process) that best describes the function you want to optimize. As the number of observations grows, the posterior distribution improves, and the algorithm becomes more certain of which regions in parameter space are worth exploring and which are not, as seen in the picture below.

As you iterate over and over, the algorithm balances its needs of exploration and exploitation taking into account what it knows about the target function. At each step a Gaussian Process is fitted to the known samples (points previously explored), and the posterior distribution, combined with a exploration strategy (such as UCB (Upper Confidence Bound), or EI (Expected Improvement)), are used to determine the next point that should be explored (see the gif below).

This process is designed to minimize the number of steps required to find a combination of parameters that are close to the optimal combination. To do so, this method uses a proxy optimization problem (finding the maximum of the acquisition function) that, albeit still a hard problem, is cheaper (in the computational sense) and common tools can be employed. Therefore Bayesian Optimization is most adequate for situations where sampling the function to be optimized is a very expensive endeavor. See the references for a proper discussion of this method.

## 1. Specifying the function to be optimized

This is a function optimization package, therefore the first and most important ingreedient is, of course, the function to be optimized.

**DISCLAIMER:** We know exactly how the output of the function below depends on its parameter. Obviously this is just an example, and you shouldn't expect to know it in a real scenario. However, it should be clear that you don't need to. All you need in order to use this package (and more generally, this technique) is a function `f` that takes a known set of parameters and outputs a real number.

In [ ]:
def black_box_function(x1, y):
    """Function with unknown internals we wish to maximize.

    This is just serving as an example, for all intents and
    purposes think of the internals of this function, i.e.: the process
    which generates its output values, as unknown.
    """
    return -x1 ** 2 - (y - 1) ** 2 + 1

## 2. Getting Started

All we need to get started is to instanciate a `BayesianOptimization` object specifying a function to be optimized `f`, and its parameters with their corresponding bounds, `pbounds`. This is a constrained optimization technique, so you must specify the minimum and maximum values that can be probed for each parameter in order for it to work

In [ ]:
!pip install bayesian-optimization==1.5.1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from bayes_opt import BayesianOptimization

In [ ]:
# Bounded region of parameter space
pbounds = {'x1': (2, 4), 'y': (-3, 3)}

In [ ]:
optimizer = BayesianOptimization(
    f=black_box_function,
    pbounds=pbounds,
    verbose=2, # verbose = 1 prints only when a maximum is observed, verbose = 0 is silent
    random_state=1,
    allow_duplicate_points=True,
)

The BayesianOptimization object will work out of the box without much tuning needed. The main method you should be aware of is `maximize`, which does exactly what you think it does.

There are many parameters you can pass to maximize, nonetheless, the most important ones are:
- `n_iter`: How many steps of bayesian optimization you want to perform. The more steps the more likely to find a good maximum you are.
- `init_points`: How many steps of **random** exploration you want to perform. Random exploration can help by diversifying the exploration space.

In [ ]:
optimizer.maximize(
    init_points=2,
    n_iter=3,
)

|   iter    |  target   |    x1     |     y     |
-------------------------------------------------
| 1         | -7.135    | 2.834     | 1.322     |
| 2         | -7.78     | 2.0       | -1.186    |
| 3         | -7.391    | 2.841     | 1.567     |
| 4         | -7.09     | 2.828     | 1.305     |
| 5         | -7.842    | 2.776     | -0.0644   |


The best combination of parameters and target value found can be accessed via the property `bo.max`.

In [ ]:
print(optimizer.max)

{'target': np.float64(-7.089552795360589), 'params': {'x1': np.float64(2.8278543171783106), 'y': np.float64(1.3046190344948578)}}


While the list of all parameters probed and their corresponding target values is available via the property `bo.res`.

In [ ]:
for i, res in enumerate(optimizer.res):
    print("Iteration {}: \n\t{}".format(i, res))

Iteration 0: 
	{'target': np.float64(-7.135455292718879), 'params': {'x1': np.float64(2.8340440094051482), 'y': np.float64(1.3219469606529488)}}
Iteration 1: 
	{'target': np.float64(-7.779531005607566), 'params': {'x1': np.float64(2.0002287496346898), 'y': np.float64(-1.1860045642089614)}}
Iteration 2: 
	{'target': np.float64(-7.390502596642655), 'params': {'x1': np.float64(2.8406441080806806), 'y': np.float64(1.5667835988004315)}}
Iteration 3: 
	{'target': np.float64(-7.089552795360589), 'params': {'x1': np.float64(2.8278543171783106), 'y': np.float64(1.3046190344948578)}}
Iteration 4: 
	{'target': np.float64(-7.841680406240952), 'params': {'x1': np.float64(2.7764615902126666), 'y': np.float64(-0.06439722111376422)}}


In [ ]:
vals = [res for i, res in enumerate(optimizer.res)]

vals

[{'target': np.float64(-7.135455292718879),
  'params': {'x1': np.float64(2.8340440094051482),
   'y': np.float64(1.3219469606529488)}},
 {'target': np.float64(-7.779531005607566),
  'params': {'x1': np.float64(2.0002287496346898),
   'y': np.float64(-1.1860045642089614)}},
 {'target': np.float64(-7.390502596642655),
  'params': {'x1': np.float64(2.8406441080806806),
   'y': np.float64(1.5667835988004315)}},
 {'target': np.float64(-7.089552795360589),
  'params': {'x1': np.float64(2.8278543171783106),
   'y': np.float64(1.3046190344948578)}},
 {'target': np.float64(-7.841680406240952),
  'params': {'x1': np.float64(2.7764615902126666),
   'y': np.float64(-0.06439722111376422)}}]

### 2.1 Changing bounds

During the optimization process you may realize the bounds chosen for some parameters are not adequate. For these situations you can invoke the method `set_bounds` to alter them. You can pass any combination of **existing** parameters and their associated new bounds.

In [ ]:
optimizer.set_bounds(new_bounds={"x": (-2, 3)})

In [ ]:
optimizer.maximize(
    init_points=0,
    n_iter=5,
)

|   iter    |  target   |    x1     |     y     |
-------------------------------------------------
| 6         | -6.503    | 2.738     | 1.087     |
| 7         | -4.752    | 2.398     | 0.9646    |
| 8         | -3.036    | 2.009     | 0.9578    |
| 9         | -3.2      | 2.0       | 1.447     |
| 10        | -31.0     | 4.0       | -3.0      |


## 3. Guiding the optimization

It is often the case that we have an idea of regions of the parameter space where the maximum of our function might lie. For these situations the `BayesianOptimization` object allows the user to specify specific points to be probed. By default these will be explored lazily (`lazy=True`), meaning these points will be evaluated only the next time you call `maximize`. This probing process happens before the gaussian process takes over.

Parameters can be passed as dictionaries such as below:

In [ ]:
optimizer.probe(
    params={"x1": 0.5, "y": 0.7},
    lazy=True,
)

Or as an iterable. Beware that the order has to be alphabetical. You can usee `optimizer.space.keys` for guidance

In [ ]:
print(optimizer.space.keys)

['x1', 'y']


In [ ]:
optimizer.probe(
    params=[-0.2, 0.1],
    lazy=True,
)

optimizer.maximize(init_points=0, n_iter=0)

for i, res in enumerate(optimizer.res):
    print("Iteration {}: \n\t{}".format(i, res))

|   iter    |  target   |    x1     |     y     |
-------------------------------------------------
| 11        | 0.66      | 0.5       | 0.7       |
| 12        | 0.15      | -0.2      | 0.1       |
Iteration 0: 
	{'target': np.float64(-7.135455292718879), 'params': {'x1': np.float64(2.8340440094051482), 'y': np.float64(1.3219469606529488)}}
Iteration 1: 
	{'target': np.float64(-7.779531005607566), 'params': {'x1': np.float64(2.0002287496346898), 'y': np.float64(-1.1860045642089614)}}
Iteration 2: 
	{'target': np.float64(-7.390502596642655), 'params': {'x1': np.float64(2.8406441080806806), 'y': np.float64(1.5667835988004315)}}
Iteration 3: 
	{'target': np.float64(-7.089552795360589), 'params': {'x1': np.float64(2.8278543171783106), 'y': np.float64(1.3046190344948578)}}
Iteration 4: 
	{'target': np.float64(-7.841680406240952), 'params': {'x1': np.float64(2.7764615902126666), 'y': np.float64(-0.06439722111376422)}}
Iteration 5: 
	{'target': np.float64(-6.502831722247593), 'params': {'x1

/usr/local/lib/python3.11/dist-packages/bayes_opt/target_space.py:376: UserWarning: 
Data point [0.5 0.7] is outside the bounds of the parameter space. 
  self.register(x, target)
/usr/local/lib/python3.11/dist-packages/bayes_opt/target_space.py:376: UserWarning: 
Data point [-0.2  0.1] is outside the bounds of the parameter space. 
  self.register(x, target)


In [ ]:
optimizer.probe(
    params=[-0.12, 0.1],
    lazy=True,
)

optimizer.maximize(init_points=0, n_iter=0)

for i, res in enumerate(optimizer.res):
    print("Iteration {}: \n\t{}".format(i, res))

|   iter    |  target   |    x1     |     y     |
-------------------------------------------------
| 13        | 0.1756    | -0.12     | 0.1       |
Iteration 0: 
	{'target': np.float64(-7.135455292718879), 'params': {'x1': np.float64(2.8340440094051482), 'y': np.float64(1.3219469606529488)}}
Iteration 1: 
	{'target': np.float64(-7.779531005607566), 'params': {'x1': np.float64(2.0002287496346898), 'y': np.float64(-1.1860045642089614)}}
Iteration 2: 
	{'target': np.float64(-7.390502596642655), 'params': {'x1': np.float64(2.8406441080806806), 'y': np.float64(1.5667835988004315)}}
Iteration 3: 
	{'target': np.float64(-7.089552795360589), 'params': {'x1': np.float64(2.8278543171783106), 'y': np.float64(1.3046190344948578)}}
Iteration 4: 
	{'target': np.float64(-7.841680406240952), 'params': {'x1': np.float64(2.7764615902126666), 'y': np.float64(-0.06439722111376422)}}
Iteration 5: 
	{'target': np.float64(-6.502831722247593), 'params': {'x1': np.float64(2.737760689287536), 'y': np.float64(

/usr/local/lib/python3.11/dist-packages/bayes_opt/target_space.py:376: UserWarning: 
Data point [-0.12  0.1 ] is outside the bounds of the parameter space. 
  self.register(x, target)


## 4. Saving, loading and restarting

By default you can follow the progress of your optimization by setting `verbose>0` when instanciating the `BayesianOptimization` object. If you need more control over logging/alerting you will need to use an observer. For more information about observers checkout the advanced tour notebook. Here we will only see how to use the native `JSONLogger` object to save to and load progress from files.

### 4.1 Saving progress

In [ ]:
import json
# Access the event log
event_log = optimizer.res  # list of dicts with 'target' and 'params'

# Save to JSON file
with open("bayes_opt_log.json", "w") as f:
    json.dump(event_log, f, indent=4)

The observer paradigm works by:
1. Instantiating an observer object.
2. Tying the observer object to a particular event fired by an optimizer.


In [ ]:
optimizer.maximize(
    init_points=0,
    n_iter=1,
)

|   iter    |  target   |    x1     |     y     |
-------------------------------------------------
| 14        | -7.0      | 2.0       | 3.0       |


### 4.2 Loading progress

Naturally, if you stored progress you will be able to load that onto a new instance of `BayesianOptimization`.

In [ ]:
with open("bayes_opt_log.json", "r") as f:
    loaded_log = json.load(f)

new_optimizer = BayesianOptimization(
    f=black_box_function,
    pbounds={"x1": (-2, 2), "y": (-2, 2)},
    verbose=2,
    random_state=7,
)
print(len(new_optimizer.space))


for entry in loaded_log:
    new_optimizer.register(
        params=entry["params"],
        target=entry["target"]
    )

0


/usr/local/lib/python3.11/dist-packages/bayes_opt/bayesian_optimization.py:225: UserWarning: 
Data point [2.83404401 1.32194696] is outside the bounds of the parameter space. 
  self._space.register(params, target, constraint_value)
/usr/local/lib/python3.11/dist-packages/bayes_opt/bayesian_optimization.py:225: UserWarning: 
Data point [ 2.00022875 -1.18600456] is outside the bounds of the parameter space. 
  self._space.register(params, target, constraint_value)
/usr/local/lib/python3.11/dist-packages/bayes_opt/bayesian_optimization.py:225: UserWarning: 
Data point [2.84064411 1.5667836 ] is outside the bounds of the parameter space. 
  self._space.register(params, target, constraint_value)
/usr/local/lib/python3.11/dist-packages/bayes_opt/bayesian_optimization.py:225: UserWarning: 
Data point [2.82785432 1.30461903] is outside the bounds of the parameter space. 
  self._space.register(params, target, constraint_value)
/usr/local/lib/python3.11/dist-packages/bayes_opt/bayesian_optimiz

In [ ]:
print("New optimizer is now aware of {} points.".format(len(new_optimizer.space)))

New optimizer is now aware of 13 points.


In [ ]:
new_optimizer.maximize(
    init_points=1,
    n_iter=1,
)

|   iter    |  target   |    x1     |     y     |
-------------------------------------------------
| 1         | -1.887    | -1.695    | 1.12      |
| 2         | -12.0     | -2.0      | -2.0      |


In [ ]:
for i, res in enumerate(new_optimizer.res):
    print("Iteration {}: \n\t{}".format(i, res))

Iteration 0: 
	{'target': np.float64(-7.135455292718879), 'params': {'x1': np.float64(2.8340440094051482), 'y': np.float64(1.3219469606529488)}}
Iteration 1: 
	{'target': np.float64(-7.779531005607566), 'params': {'x1': np.float64(2.0002287496346898), 'y': np.float64(-1.1860045642089614)}}
Iteration 2: 
	{'target': np.float64(-7.390502596642655), 'params': {'x1': np.float64(2.8406441080806806), 'y': np.float64(1.5667835988004315)}}
Iteration 3: 
	{'target': np.float64(-7.089552795360589), 'params': {'x1': np.float64(2.8278543171783106), 'y': np.float64(1.3046190344948578)}}
Iteration 4: 
	{'target': np.float64(-7.841680406240952), 'params': {'x1': np.float64(2.7764615902126666), 'y': np.float64(-0.06439722111376422)}}
Iteration 5: 
	{'target': np.float64(-6.502831722247593), 'params': {'x1': np.float64(2.737760689287536), 'y': np.float64(1.0865917457927037)}}
Iteration 6: 
	{'target': np.float64(-4.752031339776044), 'params': {'x1': np.float64(2.398078475689637), 'y': np.float64(0.9646

## Next Steps

This tour should be enough to cover most usage scenarios of this package. If, however, you feel like you need to know more, please checkout the `advanced-tour` notebook. There you will be able to find other, more advanced features of this package that could be what you're looking for. Also, browse the examples folder for implementation tips and ideas.